# 07 — Siamese V3 with Informative Negative Mining

## Obiettivo

Migliorare la Siamese Network senza modificare il task pairwise richiesto
dal tutor.

Gli esperimenti precedenti hanno mostrato che:

- CNN V3 + Euclidean Contrastive Loss è la migliore baseline Siamese;
- Cosine Similarity è risultata leggermente peggiore;
- Batch-Hard Triplet Loss ha provocato representation collapse;
- una Compact Residual CNN non ha migliorato né il pairwise ROC-AUC
  né la classificazione multiclass.

La nuova ipotesi riguarda quindi il pair sampling.

Nella baseline precedente le coppie negative erano costruite scegliendo
casualmente una delle classi diverse da quella dell'anchor.

Questo può generare molte negative pair già facilmente separabili.

In questo esperimento:

1. si carica il miglior encoder V3 già addestrato;
2. si estraggono gli embedding del training set;
3. si individuano, per ogni campione, le classi negative più vicine;
4. queste informazioni vengono utilizzate per creare coppie negative
   più informative;
5. la rete viene fine-tunata ancora tramite Euclidean Contrastive Loss.

La rete rimane una Siamese pura:

FCGR A → shared CNN V3 → embedding A
                              │
                              ├─ Euclidean Distance
                              │
FCGR B → shared CNN V3 → embedding B
                              ↓
                      Contrastive Loss

## Configurazione

- Task: 12 classi Tumor + Healthy
- FCGR: k=6
- CNN encoder: V3
- Embedding: 128D L2-normalizzato
- Positive pairs: 50%
- Negative pairs: 50%
- Informative negative fraction: 75%
- Random negative fraction: 25%
- Euclidean margin: 1.25
- Downsampling: 2111 sample/class
- Test set non utilizzato

In [1]:
# ============================================================
# CELL 2 — IMPORTS + CONFIG
# ============================================================

from pathlib import Path

import random
import time
import copy
import json

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import (
    Dataset,
    DataLoader
)

from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    f1_score,
    balanced_accuracy_score,
    confusion_matrix
)


# ============================================================
# PROJECT
# ============================================================

CURRENT_DIR = Path.cwd().resolve()

PROJECT_ROOT = (
    CURRENT_DIR.parent
    if CURRENT_DIR.name == "notebooks"
    else CURRENT_DIR
)


PROCESSED_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
)


ARTIFACTS_DIR = (
    PROJECT_ROOT
    / "artifacts"
    / "siamese_semihard_tumor_healthy"
)


ARTIFACTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# PATHS
# ============================================================

MANIFEST_PATH = (
    PROCESSED_DIR
    / "siamese_tumor_healthy_manifest.tsv"
)


CLASS_MAPPING_PATH = (
    PROCESSED_DIR
    / "siamese_tumor_healthy_class_mapping.tsv"
)


VAL_POOL_PATH = (
    PROCESSED_DIR
    / "siamese_val_pair_pool.tsv"
)


PAIR_CONFIG_PATH = (
    PROCESSED_DIR
    / "siamese_pair_config.json"
)


V3_CHECKPOINT_PATH = (
    PROJECT_ROOT
    / "artifacts"
    / "siamese_euclidean_v3_tumor_healthy"
    / "euclidean_v3_tumor_healthy_margin_1p25_best.pt"
)


# ============================================================
# PAIR CONFIG
# ============================================================

with open(
    PAIR_CONFIG_PATH,
    "r",
    encoding="utf-8"
) as f:

    pair_config = json.load(f)


K = int(
    pair_config["k"]
)

RANDOM_STATE = int(
    pair_config["random_state"]
)

TRAIN_PAIRS_PER_EPOCH = int(
    pair_config["train_pairs_per_epoch"]
)

VAL_PAIRS = int(
    pair_config["val_pairs"]
)


FCGR_PATH = (
    PROCESSED_DIR
    / "fcgr_cache"
    / f"fcgr_k{K}.npy"
)


FCGR_INDEX_PATH = (
    PROCESSED_DIR
    / "fcgr_cache"
    / f"fcgr_k{K}_index.tsv"
)


# ============================================================
# EXPERIMENT
# ============================================================

N_CLASSES = 12

EMBEDDING_DIM = 128

EUCLIDEAN_MARGIN = 1.25

POSITIVE_FRACTION = 0.50

INFORMATIVE_NEGATIVE_PROB = 0.75

HARD_TOP_K = 3

BATCH_SIZE = 128


# Fine-tuning: più basso rispetto al training from scratch
LEARNING_RATE = 1e-4

WEIGHT_DECAY = 1e-4


# ============================================================
# DEVICE
# ============================================================

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

AMP_ENABLED = (
    DEVICE.type == "cuda"
)


print("Project:", PROJECT_ROOT)
print("Device:", DEVICE)
print("k:", K)
print("V3 checkpoint:", V3_CHECKPOINT_PATH)
print("Informative negatives:", INFORMATIVE_NEGATIVE_PROB)
print("Top-k difficult classes:", HARD_TOP_K)

Project: D:\Daria\Desktop\eccdna_fcgr_siamese
Device: cuda
k: 6
V3 checkpoint: D:\Daria\Desktop\eccdna_fcgr_siamese\artifacts\siamese_euclidean_v3_tumor_healthy\euclidean_v3_tumor_healthy_margin_1p25_best.pt
Informative negatives: 0.75
Top-k difficult classes: 3


In [2]:
# ============================================================
# CELL 3 — REPRODUCIBILITY
# ============================================================

def set_seed(seed):

    random.seed(seed)

    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():

        torch.cuda.manual_seed_all(seed)


set_seed(
    RANDOM_STATE
)


if DEVICE.type == "cuda":

    torch.backends.cudnn.benchmark = True

    torch.set_float32_matmul_precision(
        "high"
    )


print(
    "Random seed:",
    RANDOM_STATE
)

Random seed: 42


In [3]:
# ============================================================
# CELL 4 — SAME 12-WAY BALANCED TRAINING SET
# ============================================================

metadata = pd.read_csv(
    MANIFEST_PATH,
    sep="\t",
    dtype={"id": str}
)


metadata["class_id"] = (
    metadata["class_id"]
    .astype(int)
)


full_train_metadata = (
    metadata[
        metadata["split_cluster"]
        ==
        "train"
    ]
    .copy()
    .reset_index(drop=True)
)


train_counts = (
    full_train_metadata[
        "class_id"
    ]
    .value_counts()
    .sort_index()
)


MIN_CLASS_SIZE = int(
    train_counts.min()
)


balanced_parts = []


for class_id in sorted(
    full_train_metadata["class_id"].unique()
):

    class_df = (
        full_train_metadata[
            full_train_metadata["class_id"]
            ==
            class_id
        ]
    )


    sampled = class_df.sample(

        n=MIN_CLASS_SIZE,

        replace=False,

        random_state=(
            RANDOM_STATE
            +
            int(class_id)
        )
    )


    balanced_parts.append(
        sampled
    )


train_metadata = (
    pd.concat(
        balanced_parts,
        ignore_index=True
    )
    .reset_index(drop=True)
)


print("=" * 72)
print("TRAIN DATA")
print("=" * 72)

print(
    "Original train:",
    len(full_train_metadata)
)

print(
    "Samples/class:",
    MIN_CLASS_SIZE
)

print(
    "Balanced train:",
    len(train_metadata)
)

print(
    "Classes:",
    train_metadata["class_id"].nunique()
)

TRAIN DATA
Original train: 96167
Samples/class: 2111
Balanced train: 25332
Classes: 12


In [4]:
# ============================================================
# CELL 5 — SAME VALIDATION POOL
# ============================================================

class_mapping = pd.read_csv(
    CLASS_MAPPING_PATH,
    sep="\t"
)


old_to_new = dict(

    zip(

        class_mapping[
            "original_class_id"
        ].astype(int),

        class_mapping[
            "class_id"
        ].astype(int)
    )
)


included_original_ids = set(
    old_to_new.keys()
)


val_original = pd.read_csv(
    VAL_POOL_PATH,
    sep="\t",
    dtype={"id": str}
)


val_original["class_id"] = (
    val_original["class_id"]
    .astype(int)
)


val_metadata = (
    val_original[
        val_original["class_id"]
        .isin(
            included_original_ids
        )
    ]
    .copy()
    .reset_index(drop=True)
)


val_metadata[
    "original_class_id"
] = (
    val_metadata[
        "class_id"
    ]
)


val_metadata[
    "class_id"
] = (

    val_metadata[
        "original_class_id"
    ]
    .map(
        old_to_new
    )
    .astype(int)
)


print(
    "Validation samples:",
    len(val_metadata)
)

print(
    "Validation classes:",
    val_metadata["class_id"].nunique()
)

Validation samples: 9753
Validation classes: 12


In [5]:
# ============================================================
# CELL 6 — FCGR CACHE
# ============================================================

fcgr_memmap = np.load(
    FCGR_PATH,
    mmap_mode="r"
)


fcgr_index = pd.read_csv(
    FCGR_INDEX_PATH,
    sep="\t",
    dtype={"id": str}
)


id_to_fcgr_row = dict(

    zip(

        fcgr_index["id"],

        fcgr_index["fcgr_row"]
    )
)


missing_train = (
    ~train_metadata["id"]
    .isin(id_to_fcgr_row)
).sum()


missing_val = (
    ~val_metadata["id"]
    .isin(id_to_fcgr_row)
).sum()


print(
    "FCGR:",
    fcgr_memmap.shape
)

print(
    "Missing train:",
    missing_train
)

print(
    "Missing val:",
    missing_val
)


assert missing_train == 0
assert missing_val == 0

FCGR: (150272, 64, 64)
Missing train: 0
Missing val: 0


In [6]:
# ============================================================
# CELL 7 — CNN ENCODER V3
# ============================================================

class FCGRCNNEncoderV3(nn.Module):

    def __init__(
        self,
        embedding_dim=128
    ):

        super().__init__()


        self.features = nn.Sequential(

            nn.Conv2d(
                1,
                32,
                3,
                padding=1,
                bias=False
            ),

            nn.GroupNorm(
                8,
                32
            ),

            nn.ReLU(
                inplace=True
            ),


            nn.Conv2d(
                32,
                32,
                3,
                padding=1,
                bias=False
            ),

            nn.ReLU(
                inplace=True
            ),

            nn.MaxPool2d(2),


            nn.Conv2d(
                32,
                64,
                3,
                padding=1,
                bias=False
            ),

            nn.GroupNorm(
                8,
                64
            ),

            nn.ReLU(
                inplace=True
            ),

            nn.MaxPool2d(2),


            nn.Conv2d(
                64,
                128,
                3,
                padding=1,
                bias=False
            ),

            nn.GroupNorm(
                8,
                128
            ),

            nn.ReLU(
                inplace=True
            ),

            nn.MaxPool2d(2),


            nn.Conv2d(
                128,
                128,
                3,
                padding=1,
                bias=False
            ),

            nn.GroupNorm(
                8,
                128
            ),

            nn.ReLU(
                inplace=True
            ),


            nn.AdaptiveAvgPool2d(
                (4, 4)
            )
        )


        self.embedding_head = nn.Sequential(

            nn.Flatten(),

            nn.Linear(
                128 * 4 * 4,
                256
            ),

            nn.ReLU(
                inplace=True
            ),

            nn.Linear(
                256,
                embedding_dim
            )
        )


    def forward(
        self,
        x
    ):

        x = self.features(x)

        z = self.embedding_head(x)


        return F.normalize(

            z,

            p=2,

            dim=1,

            eps=1e-8
        )

In [7]:
# ============================================================
# CELL 8 — SIAMESE V3 + BEST PRETRAINED CHECKPOINT
# ============================================================

class SiameseV3(nn.Module):

    def __init__(
        self,
        embedding_dim=128
    ):

        super().__init__()

        self.encoder = FCGRCNNEncoderV3(
            embedding_dim=embedding_dim
        )


    def forward(
        self,
        x1,
        x2
    ):

        batch_size = x1.shape[0]


        x = torch.cat(
            [x1, x2],
            dim=0
        )


        z = self.encoder(x)


        return (
            z[:batch_size],
            z[batch_size:]
        )


model = SiameseV3(
    embedding_dim=EMBEDDING_DIM
).to(DEVICE)


checkpoint = torch.load(

    V3_CHECKPOINT_PATH,

    map_location=DEVICE
)


print(
    "Checkpoint keys:",
    checkpoint.keys()
)


state_dict = checkpoint[
    "model_state_dict"
]


load_result = model.load_state_dict(

    state_dict,

    strict=False
)


print()
print("=" * 72)
print("V3 CHECKPOINT LOAD")
print("=" * 72)

print(
    "Checkpoint epoch:",
    checkpoint.get("epoch")
)

print(
    "Missing keys:",
    load_result.missing_keys
)

print(
    "Unexpected keys:",
    load_result.unexpected_keys
)


assert len(
    load_result.missing_keys
) == 0


assert len(
    load_result.unexpected_keys
) == 0


print()
print(
    "Best Euclidean V3 loaded: OK"
)

Checkpoint keys: dict_keys(['epoch', 'best_epoch', 'best_val_auc', 'model_state_dict', 'optimizer_state_dict', 'embedding_dim', 'margin', 'learning_rate', 'weight_decay', 'k', 'n_classes', 'task', 'val_loss', 'd_pos', 'd_neg', 'gap', 'd_prime'])

V3 CHECKPOINT LOAD
Checkpoint epoch: 19
Missing keys: []
Unexpected keys: []

Best Euclidean V3 loaded: OK


In [8]:
# ============================================================
# CELL 9 — SINGLE FCGR DATASET
# ============================================================

class SingleFCGRDataset(Dataset):

    def __init__(
        self,
        metadata,
        fcgr_memmap,
        id_to_row
    ):

        self.metadata = (
            metadata
            .copy()
            .reset_index(drop=True)
        )


        self.fcgr_memmap = (
            fcgr_memmap
        )


        self.rows = (

            self.metadata["id"]
            .astype(str)
            .map(id_to_row)
            .to_numpy(dtype=np.int64)
        )


        self.labels = (

            self.metadata["class_id"]
            .to_numpy(dtype=np.int64)
        )


    def __len__(self):

        return len(
            self.metadata
        )


    def __getitem__(
        self,
        index
    ):

        row = int(
            self.rows[index]
        )


        x = np.array(

            self.fcgr_memmap[row],

            dtype=np.float32,

            copy=True
        )


        return {

            "x":
                torch.from_numpy(x)
                .unsqueeze(0),

            "class_id":
                torch.tensor(
                    self.labels[index],
                    dtype=torch.long
                ),

            "index":
                torch.tensor(
                    index,
                    dtype=torch.long
                )
        }


embedding_dataset = SingleFCGRDataset(

    metadata=train_metadata,

    fcgr_memmap=fcgr_memmap,

    id_to_row=id_to_fcgr_row
)


embedding_loader = DataLoader(

    embedding_dataset,

    batch_size=256,

    shuffle=False,

    num_workers=0,

    pin_memory=(
        DEVICE.type == "cuda"
    )
)


print(
    "Samples for mining:",
    len(embedding_dataset)
)

Samples for mining: 25332


In [9]:
# ============================================================
# CELL 10 — EXTRACT PRETRAINED V3 EMBEDDINGS
# ============================================================

def extract_embeddings(
    encoder,
    loader
):

    encoder.eval()


    all_embeddings = []
    all_labels = []


    with torch.no_grad():

        for batch in loader:

            x = (
                batch["x"]
                .to(
                    DEVICE,
                    non_blocking=True
                )
            )


            with torch.autocast(

                device_type=DEVICE.type,

                dtype=(
                    torch.float16
                    if DEVICE.type == "cuda"
                    else torch.bfloat16
                ),

                enabled=AMP_ENABLED
            ):

                z = encoder(x)


            all_embeddings.append(

                z.float()
                .cpu()
                .numpy()
            )


            all_labels.append(

                batch["class_id"]
                .numpy()
            )


    return (

        np.concatenate(
            all_embeddings,
            axis=0
        ),

        np.concatenate(
            all_labels,
            axis=0
        )
    )


train_embeddings, train_labels = extract_embeddings(

    model.encoder,

    embedding_loader
)


print(
    "Embeddings:",
    train_embeddings.shape
)

print(
    "Labels:",
    train_labels.shape
)

print(
    "Mean norm:",
    np.linalg.norm(
        train_embeddings,
        axis=1
    ).mean()
)

Embeddings: (25332, 128)
Labels: (25332,)
Mean norm: 1.0


In [10]:
# ============================================================
# CELL 11 — CLASS PROTOTYPES
# ============================================================

prototypes = []


for class_id in range(
    N_CLASSES
):

    class_embeddings = (
        train_embeddings[
            train_labels == class_id
        ]
    )


    prototype = (
        class_embeddings.mean(
            axis=0
        )
    )


    prototype /= (
        np.linalg.norm(prototype)
        +
        1e-12
    )


    prototypes.append(
        prototype
    )


prototypes = np.stack(
    prototypes,
    axis=0
)


# ============================================================
# PROTOTYPE DISTANCES
# ============================================================

prototype_distances = np.linalg.norm(

    prototypes[:, None, :]

    -

    prototypes[None, :, :],

    axis=2
)


np.fill_diagonal(
    prototype_distances,
    np.nan
)


print(
    "Prototype distance matrix:",
    prototype_distances.shape
)

Prototype distance matrix: (12, 12)


In [11]:
# ============================================================
# CELL 12 — CLASS NAMES
# ============================================================

mapping_df = pd.read_csv(
    CLASS_MAPPING_PATH,
    sep="\t"
)


class_names = dict(

    zip(

        mapping_df[
            "class_id"
        ].astype(int),

        mapping_df[
            "disease_clean"
        ].astype(str)
    )
)


for class_id in range(
    N_CLASSES
):

    print(
        f"{class_id:2d}",
        "→",
        class_names[class_id]
    )

 0 → gastric cancer
 1 → healthy
 2 → ovarian cancer
 3 → prostate cancer
 4 → colorectal cancer
 5 → lymphoma
 6 → cervical adenocarcinoma
 7 → leukemia
 8 → hypopharyngeal squamous cell carcinoma
 9 → glioblastoma cancer
10 → melanoma
11 → hypopharynx cancer


In [12]:
# ============================================================
# CELL 13 — NEAREST NEGATIVE CLASSES
# ============================================================

nearest_rows = []


for class_id in range(
    N_CLASSES
):

    row = (
        prototype_distances[
            class_id
        ]
    )


    valid_classes = np.array(

        [
            c
            for c in range(N_CLASSES)
            if c != class_id
        ],

        dtype=int
    )


    valid_distances = (
        row[
            valid_classes
        ]
    )


    order = np.argsort(
        valid_distances
    )


    nearest_classes = (

        valid_classes[
            order[:HARD_TOP_K]
        ]
    )


    nearest_distances = (

        valid_distances[
            order[:HARD_TOP_K]
        ]
    )


    for rank, (
        negative_class,
        distance
    ) in enumerate(

        zip(
            nearest_classes,
            nearest_distances
        ),

        start=1
    ):

        nearest_rows.append(
            {
                "anchor_class":
                    class_id,

                "anchor_name":
                    class_names[
                        class_id
                    ],

                "rank":
                    rank,

                "negative_class":
                    int(
                        negative_class
                    ),

                "negative_name":
                    class_names[
                        int(
                            negative_class
                        )
                    ],

                "prototype_distance":
                    float(
                        distance
                    )
            }
        )


nearest_classes_df = pd.DataFrame(
    nearest_rows
)


display(
    nearest_classes_df
)

,anchor_class,anchor_name,rank,negative_class,negative_name,prototype_distance
0,0,gastric cancer,1,9,glioblastoma cancer,0.241462
1,0,gastric cancer,2,7,leukemia,0.278857
2,0,gastric cancer,3,4,colorectal cancer,0.310426
3,1,healthy,1,7,leukemia,0.149445
4,1,healthy,2,9,glioblastoma cancer,0.172289
5,1,healthy,3,8,hypopharyngeal squamous cell carcinoma,0.308813
6,2,ovarian cancer,1,3,prostate cancer,0.139554
7,2,ovarian cancer,2,5,lymphoma,0.360722
8,2,ovarian cancer,3,4,colorectal cancer,0.465778
9,3,prostate cancer,1,2,ovarian cancer,0.139554


In [13]:
# ============================================================
# CELL 14 — SAMPLE-SPECIFIC INFORMATIVE NEGATIVE CLASSES
# ============================================================

sample_to_prototype_distances = np.linalg.norm(

    train_embeddings[:, None, :]

    -

    prototypes[None, :, :],

    axis=2
)


informative_negative_classes = np.empty(

    (
        len(train_embeddings),
        HARD_TOP_K
    ),

    dtype=np.int64
)


for i in range(
    len(train_embeddings)
):

    true_class = int(
        train_labels[i]
    )


    distances = (
        sample_to_prototype_distances[i]
        .copy()
    )


    # Mai scegliere la classe corretta
    distances[
        true_class
    ] = np.inf


    informative_negative_classes[
        i
    ] = np.argsort(
        distances
    )[
        :HARD_TOP_K
    ]


print(
    "Informative negative matrix:",
    informative_negative_classes.shape
)


print()

print(
    "Example sample class:",
    class_names[
        int(train_labels[0])
    ]
)

print(
    "Its informative negative classes:"
)


for negative_class in (
    informative_negative_classes[0]
):

    print(
        "  -",
        class_names[
            int(negative_class)
        ]
    )

Informative negative matrix: (25332, 3)

Example sample class: gastric cancer
Its informative negative classes:
  - cervical adenocarcinoma
  - lymphoma
  - colorectal cancer


In [14]:
# ============================================================
# CELL 15 — MOST COMMON CONFUSIONS
# ============================================================

confusion_rows = []


for anchor_class in range(
    N_CLASSES
):

    sample_mask = (
        train_labels
        ==
        anchor_class
    )


    candidate_classes = (

        informative_negative_classes[
            sample_mask
        ]

        .reshape(-1)
    )


    values, counts = np.unique(

        candidate_classes,

        return_counts=True
    )


    order = np.argsort(
        counts
    )[::-1]


    for negative_class, count in zip(

        values[order],

        counts[order]
    ):

        confusion_rows.append(
            {
                "anchor_class":
                    anchor_class,

                "anchor_name":
                    class_names[
                        anchor_class
                    ],

                "negative_class":
                    int(
                        negative_class
                    ),

                "negative_name":
                    class_names[
                        int(
                            negative_class
                        )
                    ],

                "count":
                    int(
                        count
                    )
            }
        )


confusion_df = pd.DataFrame(
    confusion_rows
)


top_confusions = (

    confusion_df
    .sort_values(
        ["anchor_class", "count"],
        ascending=[
            True,
            False
        ]
    )
    .groupby(
        "anchor_class",
        as_index=False
    )
    .head(3)
)


display(
    top_confusions
)

,anchor_class,anchor_name,negative_class,negative_name,count
0,0,gastric cancer,9,glioblastoma cancer,1242
1,0,gastric cancer,7,leukemia,1096
2,0,gastric cancer,6,cervical adenocarcinoma,970
11,1,healthy,9,glioblastoma cancer,1171
12,1,healthy,7,leukemia,1097
13,1,healthy,8,hypopharyngeal squamous cell carcinoma,806
22,2,ovarian cancer,3,prostate cancer,1690
23,2,ovarian cancer,5,lymphoma,1400
24,2,ovarian cancer,11,hypopharynx cancer,1204
33,3,prostate cancer,5,lymphoma,1297


In [15]:
# ============================================================
# CELL 16 — INFORMATIVE SIAMESE PAIR DATASET
# ============================================================

class InformativeSiamesePairDataset(Dataset):

    def __init__(
        self,
        metadata,
        fcgr_memmap,
        id_to_row,
        n_pairs,
        informative_negative_classes,
        positive_fraction=0.50,
        informative_negative_prob=0.75,
        seed=42,
        dynamic=True
    ):

        self.metadata = (
            metadata
            .copy()
            .reset_index(drop=True)
        )

        self.fcgr_memmap = fcgr_memmap

        self.n_pairs = int(n_pairs)

        self.positive_fraction = float(
            positive_fraction
        )

        self.informative_negative_prob = float(
            informative_negative_prob
        )

        self.seed = int(seed)

        self.dynamic = bool(dynamic)


        # ----------------------------------------------------
        # FCGR rows + labels
        # ----------------------------------------------------

        self.rows = (
            self.metadata["id"]
            .astype(str)
            .map(id_to_row)
            .to_numpy(dtype=np.int64)
        )

        self.labels = (
            self.metadata["class_id"]
            .to_numpy(dtype=np.int64)
        )

        self.classes = np.array(
            sorted(
                np.unique(self.labels)
            ),
            dtype=np.int64
        )


        self.class_to_indices = {

            int(class_id):
                np.where(
                    self.labels == class_id
                )[0]

            for class_id in self.classes
        }


        self.informative_negative_classes = np.asarray(
            informative_negative_classes,
            dtype=np.int64
        )


        assert (
            len(self.informative_negative_classes)
            ==
            len(self.metadata)
        )


        # Più peso al negative rank 1
        self.rank_probabilities = np.array(
            [0.50, 0.30, 0.20],
            dtype=np.float64
        )

        self.rank_probabilities /= (
            self.rank_probabilities.sum()
        )


        self.epoch = 0

        self._generate_pairs(
            seed=self.seed
        )


    # ========================================================
    # PAIR GENERATION
    # ========================================================

    def _generate_pairs(
        self,
        seed
    ):

        rng = np.random.default_rng(
            seed
        )


        n_positive = int(
            round(
                self.n_pairs
                *
                self.positive_fraction
            )
        )


        targets = np.zeros(
            self.n_pairs,
            dtype=np.float32
        )

        targets[:n_positive] = 1.0

        rng.shuffle(targets)


        anchors = rng.integers(
            low=0,
            high=len(self.labels),
            size=self.n_pairs
        )


        partners = np.empty(
            self.n_pairs,
            dtype=np.int64
        )


        mining_type = np.full(
            self.n_pairs,
            "positive",
            dtype=object
        )


        for i in range(
            self.n_pairs
        ):

            anchor_idx = int(
                anchors[i]
            )

            anchor_class = int(
                self.labels[anchor_idx]
            )


            # =================================================
            # POSITIVE PAIR
            # =================================================

            if targets[i] == 1.0:

                candidates = (
                    self.class_to_indices[
                        anchor_class
                    ]
                )

                partner_idx = anchor_idx


                while (
                    partner_idx
                    ==
                    anchor_idx
                ):

                    partner_idx = int(
                        rng.choice(
                            candidates
                        )
                    )


                mining_type[i] = (
                    "positive"
                )


            # =================================================
            # NEGATIVE PAIR
            # =================================================

            else:

                use_informative = (
                    rng.random()
                    <
                    self.informative_negative_prob
                )


                # ---------------------------------------------
                # INFORMATIVE NEGATIVE
                # ---------------------------------------------

                if use_informative:

                    candidate_classes = (
                        self.informative_negative_classes[
                            anchor_idx
                        ]
                    )


                    # Supporta anche HARD_TOP_K != 3
                    probabilities = (
                        self.rank_probabilities[
                            :len(candidate_classes)
                        ]
                    )

                    probabilities = (
                        probabilities
                        /
                        probabilities.sum()
                    )


                    negative_class = int(
                        rng.choice(
                            candidate_classes,
                            p=probabilities
                        )
                    )


                    mining_type[i] = (
                        "informative"
                    )


                # ---------------------------------------------
                # RANDOM NEGATIVE
                # ---------------------------------------------

                else:

                    negative_classes = (
                        self.classes[
                            self.classes
                            !=
                            anchor_class
                        ]
                    )


                    negative_class = int(
                        rng.choice(
                            negative_classes
                        )
                    )


                    mining_type[i] = (
                        "random"
                    )


                partner_idx = int(
                    rng.choice(
                        self.class_to_indices[
                            negative_class
                        ]
                    )
                )


            partners[i] = partner_idx


        self.anchor_indices = anchors

        self.partner_indices = partners

        self.row1 = self.rows[
            anchors
        ]

        self.row2 = self.rows[
            partners
        ]

        self.targets = targets

        self.mining_type = mining_type


    # ========================================================
    # NEW PAIRS EACH EPOCH
    # ========================================================

    def set_epoch(
        self,
        epoch
    ):

        self.epoch = int(epoch)


        if self.dynamic:

            self._generate_pairs(

                seed=(
                    self.seed
                    +
                    self.epoch
                    *
                    100_003
                )
            )


    def __len__(self):

        return self.n_pairs


    def __getitem__(
        self,
        index
    ):

        row1 = int(
            self.row1[index]
        )

        row2 = int(
            self.row2[index]
        )


        x1 = np.array(
            self.fcgr_memmap[row1],
            dtype=np.float32,
            copy=True
        )

        x2 = np.array(
            self.fcgr_memmap[row2],
            dtype=np.float32,
            copy=True
        )


        return {

            "x1":
                torch.from_numpy(x1)
                .unsqueeze(0),

            "x2":
                torch.from_numpy(x2)
                .unsqueeze(0),

            "target":
                torch.tensor(
                    self.targets[index],
                    dtype=torch.float32
                )
        }

In [16]:
# ============================================================
# CELL 17 — STANDARD RANDOM PAIRS FOR VALIDATION
# ============================================================

class RandomSiamesePairDataset(Dataset):

    def __init__(
        self,
        metadata,
        fcgr_memmap,
        id_to_row,
        n_pairs,
        positive_fraction=0.5,
        seed=42
    ):

        self.metadata = (
            metadata
            .copy()
            .reset_index(drop=True)
        )

        self.fcgr_memmap = fcgr_memmap

        self.n_pairs = int(
            n_pairs
        )

        self.seed = int(
            seed
        )


        self.rows = (
            self.metadata["id"]
            .astype(str)
            .map(id_to_row)
            .to_numpy(dtype=np.int64)
        )

        self.labels = (
            self.metadata["class_id"]
            .to_numpy(dtype=np.int64)
        )


        self.classes = np.array(
            sorted(
                np.unique(
                    self.labels
                )
            ),
            dtype=np.int64
        )


        self.class_to_indices = {

            int(c):
                np.where(
                    self.labels == c
                )[0]

            for c in self.classes
        }


        rng = np.random.default_rng(
            self.seed
        )


        n_positive = int(
            round(
                self.n_pairs
                *
                positive_fraction
            )
        )


        self.targets = np.zeros(
            self.n_pairs,
            dtype=np.float32
        )

        self.targets[
            :n_positive
        ] = 1.0

        rng.shuffle(
            self.targets
        )


        anchors = rng.integers(

            0,
            len(self.labels),

            size=self.n_pairs
        )


        partners = np.empty(
            self.n_pairs,
            dtype=np.int64
        )


        for i in range(
            self.n_pairs
        ):

            anchor_idx = int(
                anchors[i]
            )

            anchor_class = int(
                self.labels[
                    anchor_idx
                ]
            )


            if self.targets[i] == 1.0:

                candidates = (
                    self.class_to_indices[
                        anchor_class
                    ]
                )

                partner_idx = (
                    anchor_idx
                )

                while (
                    partner_idx
                    ==
                    anchor_idx
                ):

                    partner_idx = int(
                        rng.choice(
                            candidates
                        )
                    )


            else:

                negative_classes = (
                    self.classes[
                        self.classes
                        !=
                        anchor_class
                    ]
                )

                negative_class = int(
                    rng.choice(
                        negative_classes
                    )
                )

                partner_idx = int(
                    rng.choice(
                        self.class_to_indices[
                            negative_class
                        ]
                    )
                )


            partners[i] = (
                partner_idx
            )


        self.row1 = self.rows[
            anchors
        ]

        self.row2 = self.rows[
            partners
        ]


    def __len__(self):

        return self.n_pairs


    def __getitem__(
        self,
        index
    ):

        x1 = np.array(

            self.fcgr_memmap[
                int(self.row1[index])
            ],

            dtype=np.float32,

            copy=True
        )


        x2 = np.array(

            self.fcgr_memmap[
                int(self.row2[index])
            ],

            dtype=np.float32,

            copy=True
        )


        return {

            "x1":
                torch.from_numpy(x1)
                .unsqueeze(0),

            "x2":
                torch.from_numpy(x2)
                .unsqueeze(0),

            "target":
                torch.tensor(
                    self.targets[index],
                    dtype=torch.float32
                )
        }

In [17]:
# ============================================================
# CELL 18 — BUILD PAIR DATASETS
# ============================================================

train_pair_dataset = (
    InformativeSiamesePairDataset(

        metadata=
            train_metadata,

        fcgr_memmap=
            fcgr_memmap,

        id_to_row=
            id_to_fcgr_row,

        n_pairs=
            TRAIN_PAIRS_PER_EPOCH,

        informative_negative_classes=
            informative_negative_classes,

        positive_fraction=
            POSITIVE_FRACTION,

        informative_negative_prob=
            INFORMATIVE_NEGATIVE_PROB,

        seed=
            RANDOM_STATE,

        dynamic=True
    )
)


val_pair_dataset = (
    RandomSiamesePairDataset(

        metadata=
            val_metadata,

        fcgr_memmap=
            fcgr_memmap,

        id_to_row=
            id_to_fcgr_row,

        n_pairs=
            VAL_PAIRS,

        positive_fraction=
            0.5,

        seed=(
            RANDOM_STATE
            +
            50_000
        )
    )
)


# ============================================================
# LOADERS
# ============================================================

train_pair_loader = DataLoader(

    train_pair_dataset,

    batch_size=BATCH_SIZE,

    shuffle=False,

    num_workers=0,

    pin_memory=(
        DEVICE.type == "cuda"
    )
)


val_pair_loader = DataLoader(

    val_pair_dataset,

    batch_size=BATCH_SIZE,

    shuffle=False,

    num_workers=0,

    pin_memory=(
        DEVICE.type == "cuda"
    )
)


# ============================================================
# DIAGNOSTIC
# ============================================================

negative_mask = (
    train_pair_dataset.targets
    ==
    0
)


negative_types = (
    train_pair_dataset.mining_type[
        negative_mask
    ]
)


informative_fraction_actual = (
    np.mean(
        negative_types
        ==
        "informative"
    )
)


random_fraction_actual = (
    np.mean(
        negative_types
        ==
        "random"
    )
)


print("=" * 72)
print("INFORMATIVE PAIR DATASET")
print("=" * 72)

print(
    "Train pairs:",
    len(train_pair_dataset)
)

print(
    "Positive fraction:",
    f"{train_pair_dataset.targets.mean():.4f}"
)

print()

print(
    "Among negative pairs:"
)

print(
    "Informative:",
    f"{informative_fraction_actual:.4f}"
)

print(
    "Random:",
    f"{random_fraction_actual:.4f}"
)

print()

print(
    "Validation pairs:",
    len(val_pair_dataset)
)

print(
    "Validation positive fraction:",
    f"{val_pair_dataset.targets.mean():.4f}"
)

INFORMATIVE PAIR DATASET
Train pairs: 50000
Positive fraction: 0.5000

Among negative pairs:
Informative: 0.7512
Random: 0.2488

Validation pairs: 10000
Validation positive fraction: 0.5000


In [18]:
# ============================================================
# CELL 19 — EUCLIDEAN CONTRASTIVE LOSS
# ============================================================

class EuclideanContrastiveLoss(nn.Module):

    def __init__(
        self,
        margin=1.25
    ):

        super().__init__()

        self.margin = float(
            margin
        )


    def forward(
        self,
        z1,
        z2,
        target
    ):

        z1 = z1.float()

        z2 = z2.float()

        target = target.float()


        distances = torch.linalg.vector_norm(

            z1 - z2,

            ord=2,

            dim=1
        )


        positive_loss = (
            target
            *
            distances.pow(2)
        )


        negative_loss = (

            (1.0 - target)

            *

            F.relu(
                self.margin
                -
                distances
            ).pow(2)
        )


        loss = (
            positive_loss
            +
            negative_loss
        ).mean()


        return (
            loss,
            distances
        )


criterion = EuclideanContrastiveLoss(

    margin=
        EUCLIDEAN_MARGIN
)

In [19]:
# ============================================================
# CELL 20 — PAIRWISE EVALUATION
# ============================================================

def evaluate_pairwise(
    model,
    loader,
    criterion
):

    model.eval()


    total_loss = 0.0
    total_samples = 0

    all_targets = []
    all_distances = []


    with torch.no_grad():

        for batch in loader:

            x1 = (
                batch["x1"]
                .to(
                    DEVICE,
                    non_blocking=True
                )
            )

            x2 = (
                batch["x2"]
                .to(
                    DEVICE,
                    non_blocking=True
                )
            )

            target = (
                batch["target"]
                .to(
                    DEVICE,
                    non_blocking=True
                )
            )


            with torch.autocast(

                device_type=
                    DEVICE.type,

                dtype=(
                    torch.float16
                    if DEVICE.type == "cuda"
                    else torch.bfloat16
                ),

                enabled=
                    AMP_ENABLED
            ):

                z1, z2 = model(
                    x1,
                    x2
                )


            loss, distances = criterion(

                z1,
                z2,
                target
            )


            batch_size = (
                target.shape[0]
            )


            total_loss += (
                loss.item()
                *
                batch_size
            )

            total_samples += (
                batch_size
            )


            all_targets.append(

                target
                .cpu()
                .numpy()
            )

            all_distances.append(

                distances
                .cpu()
                .numpy()
            )


    targets = np.concatenate(
        all_targets
    )

    distances = np.concatenate(
        all_distances
    )


    positive_distances = (
        distances[
            targets == 1
        ]
    )

    negative_distances = (
        distances[
            targets == 0
        ]
    )


    d_pos = float(
        positive_distances.mean()
    )

    d_neg = float(
        negative_distances.mean()
    )


    gap = (
        d_neg
        -
        d_pos
    )


    pooled_variance = (
        0.5
        *
        (
            positive_distances.var()
            +
            negative_distances.var()
        )
    )


    d_prime = float(

        gap

        /

        np.sqrt(
            pooled_variance
            +
            1e-12
        )
    )


    auc = float(

        roc_auc_score(

            targets,

            -distances
        )
    )


    return {

        "loss":
            total_loss
            /
            total_samples,

        "auc":
            auc,

        "d_pos":
            d_pos,

        "d_neg":
            d_neg,

        "gap":
            gap,

        "d_prime":
            d_prime
    }

In [20]:
# ============================================================
# CELL 21 — PRE-FINETUNING BASELINE
# ============================================================

pre_finetune_metrics = evaluate_pairwise(

    model=
        model,

    loader=
        val_pair_loader,

    criterion=
        criterion
)


print("=" * 76)
print("V3 BEFORE INFORMATIVE MINING FINETUNING")
print("=" * 76)

print(
    "Val ROC-AUC:",
    f"{pre_finetune_metrics['auc']:.6f}"
)

print(
    "Val loss:",
    f"{pre_finetune_metrics['loss']:.6f}"
)

print(
    "d+:",
    f"{pre_finetune_metrics['d_pos']:.6f}"
)

print(
    "d-:",
    f"{pre_finetune_metrics['d_neg']:.6f}"
)

print(
    "Gap:",
    f"{pre_finetune_metrics['gap']:.6f}"
)

print(
    "d-prime:",
    f"{pre_finetune_metrics['d_prime']:.6f}"
)

V3 BEFORE INFORMATIVE MINING FINETUNING
Val ROC-AUC: 0.661052
Val loss: 0.361724
d+: 0.572445
d-: 0.696728
Gap: 0.124283
d-prime: 0.587036


In [21]:
# ============================================================
# CELL 22 — INFORMATIVE PAIR DIFFICULTY CHECK
# ============================================================

model.eval()


all_distances = []
all_targets = []
all_mining_types = []


diagnostic_loader = DataLoader(

    train_pair_dataset,

    batch_size=BATCH_SIZE,

    shuffle=False,

    num_workers=0,

    pin_memory=(
        DEVICE.type == "cuda"
    )
)


offset = 0


with torch.no_grad():

    for batch in diagnostic_loader:

        x1 = (
            batch["x1"]
            .to(
                DEVICE,
                non_blocking=True
            )
        )

        x2 = (
            batch["x2"]
            .to(
                DEVICE,
                non_blocking=True
            )
        )

        target = (
            batch["target"]
            .to(
                DEVICE,
                non_blocking=True
            )
        )


        with torch.autocast(

            device_type=DEVICE.type,

            dtype=(
                torch.float16
                if DEVICE.type == "cuda"
                else torch.bfloat16
            ),

            enabled=AMP_ENABLED
        ):

            z1, z2 = model(
                x1,
                x2
            )


        _, distances = criterion(

            z1,
            z2,
            target
        )


        batch_size_now = (
            target.shape[0]
        )


        all_distances.append(

            distances
            .cpu()
            .numpy()
        )


        all_targets.append(

            target
            .cpu()
            .numpy()
        )


        all_mining_types.extend(

            train_pair_dataset.mining_type[
                offset:
                offset + batch_size_now
            ]
        )


        offset += (
            batch_size_now
        )


all_distances = np.concatenate(
    all_distances
)


all_targets = np.concatenate(
    all_targets
)


all_mining_types = np.asarray(
    all_mining_types
)


positive_mask = (
    all_targets == 1
)


informative_mask = (
    (all_targets == 0)
    &
    (all_mining_types == "informative")
)


random_negative_mask = (
    (all_targets == 0)
    &
    (all_mining_types == "random")
)


positive_mean = float(

    all_distances[
        positive_mask
    ].mean()
)


informative_negative_mean = float(

    all_distances[
        informative_mask
    ].mean()
)


random_negative_mean = float(

    all_distances[
        random_negative_mask
    ].mean()
)


print("=" * 76)
print("NEGATIVE MINING DIFFICULTY CHECK")
print("=" * 76)

print(
    "Positive mean distance:",
    f"{positive_mean:.6f}"
)

print(
    "Informative negative distance:",
    f"{informative_negative_mean:.6f}"
)

print(
    "Random negative distance:",
    f"{random_negative_mean:.6f}"
)

print()

print(
    "Informative gap:",
    f"{informative_negative_mean - positive_mean:.6f}"
)

print(
    "Random gap:",
    f"{random_negative_mean - positive_mean:.6f}"
)

print()


if (
    informative_negative_mean
    <
    random_negative_mean
):

    print(
        "Informative negatives are harder than random negatives: OK"
    )

else:

    print(
        "WARNING: informative negatives are not harder."
    )

NEGATIVE MINING DIFFICULTY CHECK
Positive mean distance: 0.549275
Informative negative distance: 0.534500
Random negative distance: 0.707010

Informative gap: -0.014776
Random gap: 0.157735

Informative negatives are harder than random negatives: OK


In [22]:
# ============================================================
# CELL 23 — SAVE PRE-FINETUNING STATE
# ============================================================

pre_finetuning_state = copy.deepcopy(

    model.state_dict()
)


print(
    "Pre-finetuning V3 state salvato."
)

Pre-finetuning V3 state salvato.


In [23]:
# ============================================================
# CELL 24 — FINETUNING OPTIMIZER
# ============================================================

try:

    optimizer = torch.optim.AdamW(

        model.parameters(),

        lr=LEARNING_RATE,

        weight_decay=WEIGHT_DECAY,

        fused=(
            DEVICE.type
            ==
            "cuda"
        )
    )


    fused_adamw = (
        DEVICE.type
        ==
        "cuda"
    )


except (
    RuntimeError,
    TypeError
):

    optimizer = torch.optim.AdamW(

        model.parameters(),

        lr=LEARNING_RATE,

        weight_decay=WEIGHT_DECAY
    )


    fused_adamw = False


scaler = torch.amp.GradScaler(

    "cuda",

    enabled=AMP_ENABLED
)


print(
    "Fine-tuning LR:",
    LEARNING_RATE
)

print(
    "Weight decay:",
    WEIGHT_DECAY
)

print(
    "Fused AdamW:",
    fused_adamw
)

Fine-tuning LR: 0.0001
Weight decay: 0.0001
Fused AdamW: True


In [24]:
# ============================================================
# CELL 25 — TRAIN ONE INFORMATIVE-PAIR EPOCH
# ============================================================

def train_informative_epoch(
    model,
    loader,
    dataset,
    criterion,
    optimizer,
    scaler,
    epoch
):

    model.train()


    # Nuove coppie informative a ogni epoca
    dataset.set_epoch(
        epoch
    )


    total_loss = 0.0

    total_samples = 0


    all_targets = []

    all_distances = []


    start_time = (
        time.perf_counter()
    )


    for batch in loader:

        x1 = (
            batch["x1"]
            .to(
                DEVICE,
                non_blocking=True
            )
        )


        x2 = (
            batch["x2"]
            .to(
                DEVICE,
                non_blocking=True
            )
        )


        target = (
            batch["target"]
            .to(
                DEVICE,
                non_blocking=True
            )
        )


        optimizer.zero_grad(
            set_to_none=True
        )


        with torch.autocast(

            device_type=DEVICE.type,

            dtype=(
                torch.float16
                if DEVICE.type == "cuda"
                else torch.bfloat16
            ),

            enabled=AMP_ENABLED
        ):

            z1, z2 = model(
                x1,
                x2
            )


        loss, distances = criterion(

            z1,
            z2,
            target
        )


        if AMP_ENABLED:

            scaler.scale(
                loss
            ).backward()


            scaler.step(
                optimizer
            )


            scaler.update()

        else:

            loss.backward()

            optimizer.step()


        batch_size_now = (
            target.shape[0]
        )


        total_loss += (
            loss.detach().item()
            *
            batch_size_now
        )


        total_samples += (
            batch_size_now
        )


        all_targets.append(

            target
            .detach()
            .cpu()
            .numpy()
        )


        all_distances.append(

            distances
            .detach()
            .cpu()
            .numpy()
        )


    targets = np.concatenate(
        all_targets
    )


    distances = np.concatenate(
        all_distances
    )


    positive_distances = (
        distances[
            targets == 1
        ]
    )


    negative_distances = (
        distances[
            targets == 0
        ]
    )


    train_auc = float(

        roc_auc_score(
            targets,
            -distances
        )
    )


    elapsed = (

        time.perf_counter()

        -

        start_time
    )


    return {

        "loss":
            total_loss
            /
            total_samples,

        "auc":
            train_auc,

        "d_pos":
            float(
                positive_distances.mean()
            ),

        "d_neg":
            float(
                negative_distances.mean()
            ),

        "gap":
            float(
                negative_distances.mean()
                -
                positive_distances.mean()
            ),

        "seconds":
            elapsed
    }

In [25]:
# ============================================================
# CELL 26 — INFORMATIVE NEGATIVE MINING SMOKE TEST
# ============================================================

SMOKE_EPOCHS = 3


print("=" * 112)
print("V3 + INFORMATIVE NEGATIVE MINING — SMOKE TEST")
print("=" * 112)


for epoch in range(
    1,
    SMOKE_EPOCHS + 1
):

    # ========================================================
    # TRAIN ON INFORMATIVE PAIRS
    # ========================================================

    train_metrics = train_informative_epoch(

        model=model,

        loader=train_pair_loader,

        dataset=train_pair_dataset,

        criterion=criterion,

        optimizer=optimizer,

        scaler=scaler,

        epoch=epoch
    )


    # ========================================================
    # VALIDATE ON STANDARD RANDOM PAIRS
    # ========================================================

    val_metrics = evaluate_pairwise(

        model=model,

        loader=val_pair_loader,

        criterion=criterion
    )


    print(

        f"Epoch {epoch:02d}/{SMOKE_EPOCHS}"

        f" | train loss "
        f"{train_metrics['loss']:.4f}"

        f" | train AUC "
        f"{train_metrics['auc']:.4f}"

        f" | train d+ "
        f"{train_metrics['d_pos']:.4f}"

        f" | train d- "
        f"{train_metrics['d_neg']:.4f}"

        f" | val loss "
        f"{val_metrics['loss']:.4f}"

        f" | val AUC "
        f"{val_metrics['auc']:.4f}"

        f" | val d+ "
        f"{val_metrics['d_pos']:.4f}"

        f" | val d- "
        f"{val_metrics['d_neg']:.4f}"

        f" | gap "
        f"{val_metrics['gap']:.4f}"

        f" | d' "
        f"{val_metrics['d_prime']:.4f}"

        f" | "
        f"{train_metrics['seconds']:.1f}s"
    )

V3 + INFORMATIVE NEGATIVE MINING — SMOKE TEST
Epoch 01/3 | train loss 0.3997 | train AUC 0.5418 | train d+ 0.5823 | train d- 0.6069 | val loss 0.3762 | val AUC 0.6096 | val d+ 0.6207 | val d- 0.6725 | gap 0.0518 | d' 0.3996 | 33.9s
Epoch 02/3 | train loss 0.3933 | train AUC 0.5487 | train d+ 0.5958 | train d- 0.6204 | val loss 0.3770 | val AUC 0.6059 | val d+ 0.6177 | val d- 0.6634 | gap 0.0457 | d' 0.3845 | 18.2s
Epoch 03/3 | train loss 0.3925 | train AUC 0.5500 | train d+ 0.5992 | train d- 0.6223 | val loss 0.3778 | val AUC 0.6004 | val d+ 0.6145 | val d- 0.6560 | gap 0.0416 | d' 0.3697 | 99.8s
